In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Users/lockendrakhadka222@gmail.com/databricks-pyspark-learning/Consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:

# Create widgets
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

# Read widget values
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

# Define Volume paths
base_path = f"/Volumes/source_catalog/landing/child_company_file/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

# Print paths
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# Define schemas
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

# Define table names
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

# Display configuration
print("Bronze Table: ", bronze_table)
print("Silver Table: ", silver_table)
print("Gold Table: ", gold_table)

# Create the processed directory if it doesn't exist
dbutils.fs.mkdirs(processed_path)


### Bronze

In [0]:
df = (
    spark.read
    .options(header=True, inferSchema=True)
    .csv(f"{landing_path}*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

print("Total Rows:", df.count())
df.show(5, truncate=False)


**Write Delta Table**

In [0]:
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

### Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

### Transformations

In [0]:
# 1. Remove rows where order_qty is NULL
df_orders = df_orders.filter(
    F.col("order_qty").isNotNull()
)

# 2. Clean customer_id
# Keep numeric IDs; replace invalid values with "999999"
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(
        F.col("customer_id").rlike("^[0-9]+$"),
        F.col("customer_id")
    ).otherwise("999999").cast("string")
)

# 3. Remove weekday names from order_placement_date
# Example: Tuesday, July 01, 2025 -> July 01, 2025
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(
        F.col("order_placement_date"),
        r"^[A-Za-z]+,\s*",
        ""
    )
)

# 4. Convert order_placement_date into a proper date
# Try multiple formats to handle inconsistent source data
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy")
    )
)

# 5. Remove duplicate orders using the five specified columns
df_orders = df_orders.dropDuplicates([
    "order_id",
    "order_placement_date",
    "customer_id",
    "product_id",
    "order_qty"
])

# 6. Convert product_id to string
df_orders = df_orders.withColumn(
    "product_id",
    F.col("product_id").cast("string")
)


In [0]:
# Check the minimum and maximum order date

df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
display(df_orders.limit(20))

In [0]:
df_products = spark.table("fmcg.silver.products")
display(df_products.limit(5))

### Join with products



In [0]:
df_joined = df_orders.join(
    df_products,
    on="product_id",
    how="inner"
).select(
    df_orders["*"],
    df_products["product_code"]
)

display(df_joined.limit(10))

In [0]:
# Check whether the Silver orders table exists
if not spark.catalog.tableExists(silver_table):

    # Create the Silver table for the first time
    df_joined.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(silver_table)

else:

    # Access the existing Silver Delta table
    silver_delta = DeltaTable.forName(spark, silver_table)

    # Merge incoming orders with existing Silver records
    silver_delta.alias("silver").merge(
        df_joined.alias("source"),
        """
        silver.order_placement_date = source.order_placement_date
        AND silver.order_id = source.order_id
        AND silver.product_code = source.product_code
        AND silver.customer_id = source.customer_id
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

# Verify the saved Silver table
df_silver_orders = spark.table(silver_table)

print("Silver table:", silver_table)
print("Total rows:", df_silver_orders.count())

display(df_silver_orders.limit(10))


### Gold

In [0]:
df_gold = spark.sql(f"""
    SELECT
        order_id,
        order_placement_date AS date,
        customer_id AS customer_code,
        product_code,
        product_id,
        order_qty AS sold_quantity
    FROM {silver_table}
""")

df_gold.show(2)

In [0]:

if not spark.catalog.tableExists(gold_table):
    print("Creating New Table")

    df_gold.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

else:
    gold_delta = DeltaTable.forName(spark, gold_table)

    gold_delta.alias("target").merge(
        df_gold.alias("source"),
        """
        target.date = source.date
        AND target.order_id = source.order_id
        AND target.product_code = source.product_code
        AND target.customer_code = source.customer_code
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()


## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

**Full Load**

In [0]:
df_child = spark.sql(
    f"""
    SELECT
        date,
        product_code,
        customer_code,
        sold_quantity
    FROM {gold_table}
    """
)

df_child.show(10)

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child

    # 1. Convert each date to the first day of its month
    .withColumn(
        "month_start",
        F.trunc(F.col("date"), "MM")
    )

    # 2. Group by month, product, and customer
    .groupBy(
        "month_start",
        "product_code",
        "customer_code"
    )

    # 3. Calculate the total quantity sold
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 4. Rename month_start to date
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)


In [0]:
df_monthly.count()

In [0]:
# Access the existing parent Gold fact table
gold_parent_delta = DeltaTable.forName(
    spark,
    f"{catalog}.{gold_schema}.fact_orders"
)

# Merge monthly child-company data into the existing Gold table
gold_parent_delta.alias("parent_gold").merge(
    df_monthly.alias("child_gold"),
    """
    parent_gold.date = child_gold.date
    AND parent_gold.product_code = child_gold.product_code
    AND parent_gold.customer_code = child_gold.customer_code
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
